1. Imports and load

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("nb06_all_spreads.csv", parse_dates=["month_date"])
print("Shape:", df.shape)
print("Date range:", df["month_date"].min(), "→", df["month_date"].max())
print("Columns:", df.columns.tolist())

Shape: (137, 24)
Date range: 2015-01-01 00:00:00 → 2026-05-01 00:00:00
Columns: ['month_date', 'cpo_price', 'soyoil_price', 'sunflower_price', 'rapeseed_price', 'brent_crude_usd', 'usd_myr', 'usd_idr', 'usd_inr', 'usd_cny', 'myr_indexed', 'idr_indexed', 'pogo_spread', 'pogo_zone', 'cpo_zscore', 'price_cycle_position', 'cpo_vs_soy_spread', 'cpo_vs_sunflower_spread', 'cpo_vs_rapeseed_spread', 'substitution_risk', 'industrial_consumption_1000mt', 'exports_1000mt', 'biodiesel_share_pct', 'fao_veg_oil_index']


2. Date continuity check

In [2]:
# Check for any gaps in the monthly sequence
expected = pd.date_range(
    start=df["month_date"].min(),
    end=df["month_date"].max(),
    freq="MS"
)
missing = expected.difference(df["month_date"])
print("Missing months:", len(missing))
print(missing)

Missing months: 0
DatetimeIndex([], dtype='datetime64[us]', freq='MS')


3. Column completeness summary

In [3]:
print(df.isnull().sum())
print()
print("Expected nulls:")
print("  cpo_zscore: 35 (36-month rolling burn-in)")
print("  USDA columns: 9 (Jan-Sep 2015 before marketing year starts)")
print("  All others: 0")

month_date                        0
cpo_price                         0
soyoil_price                      0
sunflower_price                   0
rapeseed_price                    0
brent_crude_usd                   0
usd_myr                           0
usd_idr                           0
usd_inr                           0
usd_cny                           0
myr_indexed                       0
idr_indexed                       0
pogo_spread                       0
pogo_zone                         0
cpo_zscore                       35
price_cycle_position              0
cpo_vs_soy_spread                 0
cpo_vs_sunflower_spread           0
cpo_vs_rapeseed_spread            0
substitution_risk                 0
industrial_consumption_1000mt     9
exports_1000mt                    9
biodiesel_share_pct               9
fao_veg_oil_index                 0
dtype: int64

Expected nulls:
  cpo_zscore: 35 (36-month rolling burn-in)
  USDA columns: 9 (Jan-Sep 2015 before marketing year starts)


4. Sanity check key values

In [6]:
# Spot check known values against raw sources
checks = [
    ("May 2026 CPO price",        "2026-05-01", "cpo_price",         1139.94),
    ("May 2026 POGO spread",      "2026-05-01", "pogo_spread",        467.97),
    ("May 2026 FAO veg oil index","2026-05-01", "fao_veg_oil_index",  185.00),
    ("May 2026 soy spread",       "2026-05-01", "cpo_vs_soy_spread", -635.32),
]

print(f"{'Check':<35} {'Expected':>10} {'Actual':>10} {'Match':>6}")
print("-" * 65)
for label, date, col, expected in checks:
    actual = df.loc[df["month_date"] == date, col].values[0]
    match = "yes" if abs(actual - expected) < 1.0 else "no"
    print(f"{label:<35} {expected:>10.2f} {actual:>10.2f} {match:>6}")

Check                                 Expected     Actual  Match
-----------------------------------------------------------------
May 2026 CPO price                     1139.94    1139.94    yes
May 2026 POGO spread                    467.97     467.97    yes
May 2026 FAO veg oil index              185.00     185.00    yes
May 2026 soy spread                    -635.32    -635.32    yes


5. POGO zone distribution

In [7]:
print("POGO zone distribution:")
print(df["pogo_zone"].value_counts())
print()
print("Substitution risk distribution:")
print(df["substitution_risk"].value_counts())
print()
print("Price cycle distribution:")
print(df["price_cycle_position"].value_counts())

POGO zone distribution:
pogo_zone
COSTLY      119
MARGINAL     18
Name: count, dtype: int64

Substitution risk distribution:
substitution_risk
LOW         93
MODERATE    26
HIGH        18
Name: count, dtype: int64

Price cycle distribution:
price_cycle_position
FAIR                 54
INSUFFICIENT DATA    35
EXPENSIVE            28
CHEAP                20
Name: count, dtype: int64


6. Final confirmation

In [8]:
print("=" * 50)
print("NB07 VALIDATION COMPLETE")
print("=" * 50)
print(f"Total months:       {len(df)}")
print(f"Date range:         {df['month_date'].min().strftime('%Y-%m')} → {df['month_date'].max().strftime('%Y-%m')}")
print(f"Missing months:     0")
print(f"Unexpected nulls:   0")
print(f"Columns validated:  {len(df.columns)}")
print()
print("nb06_all_spreads.csv is ready for pipeline and dbt development.")

NB07 VALIDATION COMPLETE
Total months:       137
Date range:         2015-01 → 2026-05
Missing months:     0
Unexpected nulls:   0
Columns validated:  24

nb06_all_spreads.csv is ready for pipeline and dbt development.
